## Build a Variational Autoencoder using Yield Curve Data

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from adjustText import adjust_text

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
# tqdm
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Prepare the data

Download the data from the Bank of England if needed.

In [ ]:
import requests, zipfile, io
zip_file_url = "https://www.bankofengland.co.uk/-/media/boe/files/statistics/yield-curves/oisddata.zip"

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

with requests.get(zip_file_url, headers=headers, stream=True) as r:
    r.raise_for_status()
    with open("oisddata.zip", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

zip_path = "oisddata.zip"
extract_path = ""

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

The data is supplied in three Excel files, covering 2009-2015, 2016-2024 and 2025 onwards.

In [ ]:
df1 = pd.read_excel("OIS daily data_2009 to 2015.xlsx", 
                    index_col=0, header=3, sheet_name="2. spot curve", skiprows=[4],
                    engine='openpyxl')
df1.dropna(inplace=True)
df2 = pd.read_excel("OIS daily data_2016 to 2024.xlsx", 
                    index_col=0, header=3, sheet_name="3. spot, short end", skiprows=[4],
                    engine='openpyxl')
df2.dropna(inplace=True)
df3 = pd.read_excel("OIS daily data_2025 to present.xlsx", 
                    index_col=0, header=3, sheet_name="3. spot, short end", skiprows=[4],
                    engine='openpyxl')
df3.dropna(inplace=True)
df = pd.concat([df1, df2, df3])

In [ ]:
df.head()

In [ ]:
X = df.to_numpy()

In [ ]:
X_train, X_test = train_test_split(X, test_size=0.10, random_state=42)

In [ ]:
scaler = StandardScaler()

In [ ]:
X_train = scaler.fit_transform(X_train)

In [ ]:
X_test = scaler.transform(X_test)

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
dataset = TensorDataset(torch_tensor)
batch_size = 128
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_dataset = TensorDataset(test_torch_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Build the Variational Autoencoder Model

In [ ]:
class Sampling(nn.Module):
    def forward(self, inputs):
        mu, log_var = inputs
        batch = mu.shape[0]
        dim = mu.shape[1]
        epsilon = torch.randn((batch, dim), device=device)
        return mu + torch.exp(0.5 * log_var) * epsilon

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, z_dim, layer_dims, use_batch_norm, use_dropout):
        super(Encoder, self).__init__()

        self.use_batch_norm = use_batch_norm
        self.use_dropout = use_dropout
        
        self.hidden_layers = nn.ModuleList()
        prev_dim = input_dim
        for i in layer_dims:
            self.hidden_layers.append(nn.Linear(prev_dim, i))
            if use_batch_norm:
                self.hidden_layers.append(nn.BatchNorm1d(i))
            self.hidden_layers.append(nn.LeakyReLU())
            if use_dropout:
                self.hidden_layers.append(nn.Dropout(p = 0.25))
            prev_dim = i

        self.mu = nn.Linear(layer_dims[-1], z_dim)
        self.log_var = nn.Linear(layer_dims[-1], z_dim)
        self.sampling = Sampling()

    def forward(self, x):
        for layer in self.hidden_layers:
            x = layer(x)
        mu = self.mu(x)
        log_var = self.log_var(x)
        z = self.sampling((mu, log_var))
        return mu, log_var, z

In [ ]:
class Decoder(nn.Module):
    def __init__(self, input_dim, z_dim, layer_dims, use_batch_norm, use_dropout):
        super(Decoder, self).__init__()

        self.use_batch_norm = use_batch_norm
        self.use_dropout = use_dropout

        self.hidden_layers = nn.ModuleList()
        prev_dim = z_dim
        for ilayer in layer_dims:
            self.hidden_layers.append(nn.Linear(prev_dim, ilayer))
            if use_batch_norm:
                self.hidden_layers.append(nn.BatchNorm1d(ilayer))
            self.hidden_layers.append(nn.LeakyReLU())
            if use_dropout:
                self.hidden_layers.append(nn.Dropout(p = 0.25))
            prev_dim = ilayer

        self.out = nn.Linear(layer_dims[-1], input_dim)

    def forward(self, x):
        for layer in self.hidden_layers:
            x = layer(x)
        x = self.out(x)
        return x

In [ ]:
input_dim = 60
z_dim = 2
encoder_layer_dims = [64, 32]
decoder_layer_dims = [32, 64]
use_batch_norm = True
use_dropout = True
learning_rate = 0.001
epochs=1000

In [ ]:
class VAE(nn.Module):
    def __init__(self, encoder, decoder, beta = 1.0):
        super(VAE, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

    def forward(self, x):
        mu, log_var, z = self.encoder(x)
        x_reconstructed = self.decoder(z)
        return mu, log_var, z, x_reconstructed

    def train_step(self, data, optimizer):
        optimizer.zero_grad()
        mu, log_var, z, reconstruction = self(data)
        reconstruction_loss = F.mse_loss(data, reconstruction, reduction='sum').mean()
        kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1).mean()
        total_loss = reconstruction_loss + self.beta * kl_loss
        total_loss.backward()
        optimizer.step()
        return total_loss, reconstruction_loss, kl_loss

In [ ]:
def train_vae(model, data_loader, optimizer, num_epochs):
    model.train()
    tqdm_epoch = trange(num_epochs)
    for epoch in tqdm_epoch:
        epoch_total_loss = 0.0
        epoch_reconstruction_loss = 0.0
        epoch_kl_loss = 0.0

        for [batch] in data_loader:
            total_loss, reconstruction_loss, kl_loss = model.train_step(batch, optimizer)
            epoch_total_loss += total_loss.item()
            epoch_reconstruction_loss += reconstruction_loss.item()
            epoch_kl_loss += kl_loss.item()

        epoch_total_loss /= len(data_loader)
        epoch_reconstruction_loss /= len(data_loader)
        epoch_kl_loss /= len(data_loader)
        tqdm_epoch.set_description('Total Loss: {:4f}, \
                                   Reconstruction Loss: {:4f}, \
                                   KL Loss: {:4f}'.format(epoch_total_loss, epoch_reconstruction_loss, epoch_kl_loss))

In [ ]:
encoder = Encoder(input_dim, z_dim, encoder_layer_dims, use_batch_norm, use_dropout)

In [ ]:
print(encoder)

In [ ]:
decoder = Decoder(input_dim, z_dim, decoder_layer_dims, use_batch_norm, use_dropout)

In [ ]:
print(decoder)

In [ ]:
beta = 10
vae = VAE(encoder, decoder, beta=beta).to(device)

In [ ]:
optimizer = optim.Adam(vae.parameters(), lr=learning_rate)

In [ ]:
train_vae(vae, data_loader, optimizer, epochs)

### Testing

In [ ]:
test_z = encoder(test_torch_tensor)

In [ ]:
test_pred = decoder(test_z[2])

In [ ]:
reconstruction_loss = torch.square(test_torch_tensor - test_pred)
reconstruction_loss = torch.mean(torch.sum(reconstruction_loss, dim=1))

In [ ]:
print(reconstruction_loss.item())

In [ ]:
test_pred

In [ ]:
test_torch_tensor

### Sampling

In [ ]:
sample_z = torch.randn((X_test.shape[0], 2), device=device)

In [ ]:
sample_curves = decoder(sample_z)

In [ ]:
sample_curves = scaler.inverse_transform(sample_curves.detach().cpu().numpy())

In [ ]:
ntest_z = test_z[2].detach().cpu().numpy()
nsample_z = sample_z.detach().cpu().numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))

ax.scatter(ntest_z[:,0], ntest_z[:,1], color= 'black', marker='+', label='test set')
ax.scatter(nsample_z[:,0], nsample_z[:,1], color= 'black', marker='o', facecolors='none', label='samples')
ax.set_xlabel('latent dim 0')
ax.set_ylabel('latent dim 1')
ax.legend()
plt.savefig('ycvaelatent.png', dpi=300, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(15,10))

# Define line styles and markers
styles = ['-', '--', '-.', ':']
markers = ['o', 'v', '^', '<', '>', 's', 'p', '*', 'h']

for i in range(10):
    style = styles[i % len(styles)]
    marker = markers[i % len(markers)]
    ax.plot(df.columns, sample_curves[i,:], linestyle=style, marker=marker, color='black')

ax.set_xlabel('Maturity')
ax.set_ylabel('Rate')
plt.savefig('ycvaesasmple.png', dpi=300, bbox_inches='tight')

sample every 60th row from original dataframe

In [ ]:
new_df = df.iloc[::126, :]
dts = new_df.index

In [ ]:
qX = new_df.to_numpy()
qX = scaler.fit_transform(qX)
qX_tensor = torch.from_numpy(qX).float().to(device)

encode

In [ ]:
q_z = encoder(qX_tensor)

In [ ]:
nq_z = q_z[2].detach().cpu().numpy()

In [ ]:
def plot_line(coords, dates, filename):
    # Ensure the coordinates are a numpy array
    coords = np.array(coords)

    # Create the plot
    plt.figure(figsize=(15, 15))
    plt.plot(coords[:, 0], coords[:, 1], '--o',  color='black')

     # Initialize list for adjust_text
    texts = list()
    
    # Label each point
    for i, date in enumerate(dates):
        texts.append(plt.text(coords[i, 0], coords[i, 1], str(date.date())))
        
    # Adjust labels to avoid overlap
    adjust_text(texts, force_points=0.2, force_text=0.2, 
                expand_points=(1,1), expand_text=(1,1), 
                arrowprops=dict(arrowstyle="-", lw=1.5, alpha=0.5))

    # Set title and labels if necessary
    plt.xlabel('X')
    plt.ylabel('Y')

    # Save the plot to a file
    plt.savefig(filename, dpi=300)

    # Show the plot
    plt.show()

In [ ]:
filename = 'vaepath.png'
plot_line(nq_z, dts, filename)